# 문제해결 1: 금융사기 보고서 기반 FAISS Naive RAG

## 실습 목적

Naive RAG의 기본 흐름을 익힌 후, **직접 문서 기반 질의응답 시스템을 구성하고 문제를 점검**하도록 설계한 실습 예제

사용 문서: `data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf`

## 실습 시나리오

금융소비자 교육 담당자가 하나의 금융사기 보고서를 기반으로 질문에 답변하는 내부용 QA 챗봇을 만들려고 한다.  

## 완성 기준

| 기준 | 설명 |
|---|---|
| 문서 로딩 | PDF 페이지를 `Document` 객체로 정상 로딩한다. |
| 청킹 | 문서가 적절한 크기의 청크로 분할된다. |
| 벡터DB 생성 | FAISS 벡터스토어가 생성되고 로컬 저장까지 가능하다. |
| 검색 | 질문과 관련된 상위 문서 조각을 검색한다. |
| 답변 생성 | 검색된 문서에 근거해서만 답변한다. |

## 0. 설치

> `faiss-cpu`는 로컬 CPU 환경에서 FAISS를 사용하기 위한 패키지이다.

In [1]:
# uv add langchain langchain-openai langchain-community langchain-text-splitters pypdf faiss-cpu python-dotenv pandas

## 1. 라이브러리 불러오기

LangChain의 기본 구성요소를 사용한다.

- `PyPDFLoader`: PDF 문서 로딩
- `RecursiveCharacterTextSplitter`: 문서 청킹
- `OpenAIEmbeddings`: 텍스트 임베딩
- `FAISS`: 로컬 벡터스토어
- `ChatOpenAI`: 답변 생성 LLM

In [2]:
from pathlib import Path
import os
from pprint import pprint

import pandas as pd
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

import sys
sys.path.append("../")
from common_config import get_llm, embedding_model

/tmp/ipykernel_77856/3241142176.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 2. 환경변수 설정

OpenAI API Key는 `.env` 파일 또는 운영체제 환경변수에 설정한다.

`.env` 파일 예시는 다음과 같다.

```text
OPENAI_API_KEY=sk-...
```

In [3]:
load_dotenv(override=True, dotenv_path="../.env")

if not os.getenv("LLM_API_KEY"):
    raise ValueError(
        "LLM_API_KEY가 설정되어 있지 않습니다. .env 파일 또는 환경변수에 LLM_API_KEY를 설정하세요."
    )

print("LLM_API_KEY 설정 확인 완료")

LLM_API_KEY 설정 확인 완료


## 3. PDF 파일 경로 설정

노트북과 같은 폴더 또는 `data` 폴더에 PDF를 넣어두면 자동으로 찾는다.

권장 폴더 구조는 다음과 같다.

```text
project/
├─ 1_faiss_naive_rag_financial_fraud.ipynb
└─ data/
   └─ 진화하는 금융사기, 모두의 경계가 필요한 시점.pdf
```

In [4]:
PDF_NAME = "진화하는 금융사기, 모두의 경계가 필요한 시점.pdf"

candidate_paths = [
    Path(PDF_NAME),
    Path("data") / PDF_NAME,
    Path("/mnt/data") / PDF_NAME,  # ChatGPT 작업환경용 경로
]

PDF_PATH = next((path for path in candidate_paths if path.exists()), None)

if PDF_PATH is None:
    raise FileNotFoundError(
        f"PDF 파일을 찾을 수 없습니다. 다음 위치 중 하나에 파일을 넣어주세요: {candidate_paths}"
    )

print("PDF_PATH:", PDF_PATH)

PDF_PATH: data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf


## 4. PDF 로딩

PDF를 페이지 단위 `Document`로 로딩한다.  
각 페이지는 `page_content`와 `metadata`를 가진다.

### pdf 파일 로딩

In [5]:
# PDF 로드
loader = PyPDFLoader(str(PDF_PATH))

# PDF 페이지를 로드하여 문서 객체 리스트로 반환
pages = loader.load()

# 메타데이터 보강
for doc in pages:
    doc.metadata["source_name"] = PDF_NAME
    doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1

print("로드된 페이지 수:", len(pages))
print("첫 번째 페이지 메타데이터:")
pprint(pages[0].metadata)
print("\n첫 번째 페이지 내용 일부:")
print(pages[0].page_content[:500])

로드된 페이지 수: 15
첫 번째 페이지 메타데이터:
{'author': 'user',
 'creationdate': '2026-05-29T09:03:29+09:00',
 'creator': 'Hancom PDF 1.3.0.534',
 'moddate': '2026-05-29T09:03:29+09:00',
 'page': 0,
 'page_label': 1,
 'pdfversion': '1.4',
 'producer': 'Hancom PDF 1.3.0.534',
 'source': 'data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'source_name': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'total_pages': 15}

첫 번째 페이지 내용 일부:
2026년 5월 29일 제 17 호
진화하는 금융사기,
모두의 경계가 필요한 시점


## 5. 문서 청킹

RAG에서는 문서 전체를 한 번에 넣지 않고, 검색 가능한 단위로 나눈다.  
 RecursiveCharacterTextSplitter()을 사용한다.  
이 실습에서는 보고서형 PDF에 적합한 기본값으로 `chunk_size=800`, `chunk_overlap=120`을 사용한다.

- `chunk_size`: 하나의 문서 조각 최대 길이
- `chunk_overlap`: 앞뒤 청크가 겹치는 길이
- 목적: 문맥 손실을 줄이면서 검색 단위를 적절히 유지하는 것

In [6]:
# 텍스트 분할 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
)

# 페이지 단위로 로드된 문서를 청크 단위로 분할
chunks = text_splitter.split_documents(pages)

# 청크 단위로 메타데이터 보강
for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i
    doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1
    doc.metadata["source_name"] = PDF_NAME

print("전체 청크 수:", len(chunks))
print("첫 번째 청크 메타데이터:")
pprint(chunks[0].metadata)
print("\n첫 번째 청크 내용:")
print(chunks[0].page_content[:800])

전체 청크 수: 27
첫 번째 청크 메타데이터:
{'author': 'user',
 'chunk_id': 0,
 'creationdate': '2026-05-29T09:03:29+09:00',
 'creator': 'Hancom PDF 1.3.0.534',
 'moddate': '2026-05-29T09:03:29+09:00',
 'page': 0,
 'page_label': 1,
 'pdfversion': '1.4',
 'producer': 'Hancom PDF 1.3.0.534',
 'source': 'data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'source_name': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'total_pages': 15}

첫 번째 청크 내용:
2026년 5월 29일 제 17 호
진화하는 금융사기,
모두의 경계가 필요한 시점


## 6. 청크 분포 확인

페이지별 청크 수를 확인한다.  
특정 페이지의 청크가 지나치게 많거나 적으면 청킹 전략을 조정해야 한다.

In [7]:
# 청크 정보를 DataFrame으로 정리하여 페이지별 청크 수와 샘플을 확인
chunk_df = pd.DataFrame([
    {
        "chunk_id": doc.metadata["chunk_id"],
        "page": doc.metadata.get("page_label"),
        "text_length": len(doc.page_content),
        "preview": doc.page_content[:80].replace("\n", " ")
    }
    for doc in chunks
])

print("페이지별 청크 수")
display(chunk_df.groupby("page").size().reset_index(name="chunk_count"))

print("청크 샘플")
display(chunk_df.head(10))

페이지별 청크 수


,page,chunk_count
0,1,1
1,2,2
2,3,2
3,4,2
4,5,2
5,6,2
6,7,2
7,8,2
8,9,2
9,10,2


청크 샘플


,chunk_id,page,text_length,preview
0,0,1,45,"2026년 5월 29일 제 17 호 진화하는 금융사기, 모두의 경계가 필요한 시점"
1,1,2,797,하나은행 ...
2,2,2,358,l 최근에는 AI로 목소리나 화면을 조작하여 피해자가 비대면으로 진위 여부를 확인할...
3,3,3,776,Hana Bank Hana Institute of Finance 2 ∎ 규제 공백 ...
4,4,3,266,측면에서 금융회사와 정부 기관의 예방 노력이 함께 이행되어야 할 필요 l 개별법 제...
5,5,4,788,하나은행 ...
6,6,4,415,피해자가 범죄에 연루되는 것을 인지하지 못하도록 인력을 동원할 수 있다는 난점 ● ...
7,7,5,791,Hana Bank Hana Institute of Finance 4 2. 금융사기 ...
8,8,5,424,"사기 행위를 분류하며, 수법에 따라 관련 규정을 지속적으로 확대하는 추세 표2 | ..."
9,9,6,766,하나은행 ...


## 7. 임베딩 모델과 FAISS 벡터스토어 생성

여기서부터 인덱싱 단계이다.

```text
청크 텍스트 → 임베딩 벡터 → FAISS 인덱스 저장
```

FAISS는 벡터 간 유사도 검색을 빠르게 수행하기 위한 로컬 벡터 검색 라이브러리이다.  
이 노트북에서는 LangChain의 `FAISS.from_documents()`를 사용해 문서 청크와 임베딩 모델을 연결한다.

In [8]:
# 1. 임베딩 모델 초기화
embeddings = embedding_model()

# 2. FAISS 인덱스 저장 경로 설정
INDEX_DIR = Path("faiss_financial_fraud_pdf_index")
FAISS_INDEX_FILE = INDEX_DIR / "index.faiss"
FAISS_PKL_FILE = INDEX_DIR / "index.pkl"

# 3. 기존 인덱스가 있으면 로딩, 없으면 새로 생성
if FAISS_INDEX_FILE.exists() and FAISS_PKL_FILE.exists():
    print("기존 FAISS 인덱스를 로딩합니다.")

    # FAISS 벡터스토어 로딩 
    vectorstore = FAISS.load_local(
        str(INDEX_DIR),
        embeddings,
        allow_dangerous_deserialization=True,
    )

    print("FAISS 인덱스 로딩 완료:", INDEX_DIR.resolve())

else:
    print("기존 FAISS 인덱스가 없습니다.")
    print("청크 문서를 임베딩하여 새 FAISS 벡터스토어를 생성합니다.")

    # FAISS 벡터스토어 생성
    vectorstore = FAISS.from_documents(chunks, embeddings)

    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(INDEX_DIR))

    print("FAISS 인덱스 생성 및 저장 완료:", INDEX_DIR.resolve())

# 4. 인덱스 상태 확인
print("저장된 문서 청크 수:", vectorstore.index.ntotal)


기존 FAISS 인덱스를 로딩합니다.
FAISS 인덱스 로딩 완료: /home/dahyun/gg1th_rag_agent_ex/2.naive_rag/faiss_financial_fraud_pdf_index
저장된 문서 청크 수: 27


## 8. 기본 유사도 검색 테스트

먼저 LLM 답변 생성 없이, 검색 결과만 확인한다.  
이 단계가 중요하다. 검색이 잘못되면 답변도 잘못될 가능성이 높다.

`similarity_search_with_score()`의 score는 기본 FAISS 설정에서는 거리 기반 점수로 해석한다.  
일반적으로 **낮을수록 더 가까운 문서**로 볼 수 있다.

In [9]:
question = "금융사기는 어떤 단계로 발생하나요?"

results = vectorstore.similarity_search_with_score(question, k=4)

for rank, (doc, score) in enumerate(results, start=1):
    print(f"\n[{rank}] score={score:.4f} | page={doc.metadata.get('page_label')} | chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:500].replace("\n", " "))


[1] score=0.8488 | page=13 | chunk=24
● 금융거래 데이터뿐만 아니라 앱 이용 패턴 데이터, 통신 데이터 등 사기 거래 전반에  필요한 데이터를 종합적으로 분석하고 학습하여 사기를 예방할 수 있는 시스템 구축 ● 금융회사가 자체적으로 개발한 금융사기 탐지 프로그램을 타 금융사와 공유하는 등  사회적 문제 해결을 위한 공유 체계를 적극적으로 마련하여 피해를 예방하고자 노력 ■ 금융사기는 개인이 순간적으로 사기를 인지하고 예방하기 어렵다는 측면에서  금융회사와 관련 정부 기관의 예방 노력이 함께 이행되어야 함 ● 개별법 제·개정, 규제 강화 중심의 예방 노력은 후행적 조치라는 한계가 존재하므로   금융사기 예방을 위한 범기관 네트워크 구축이 요구 - 통신사, 가상자산거래소, 은행 등 금융사기가 발생하는 지점과 직접적으로 관련된 기관과의  공유 네트워크를 구축함으로써 민관 협력을 통한 예방 체계를 고도화 ● 금융사는 금융사기 전담 부서 조직을 확충하고 금융사기를 예방할 수 있는 시스템을  운영하는 한편, 피해 발생 시 배상 프

[2] score=0.8891 | page=4 | chunk=5
하나은행                                                                                                                    하나금융연구소 33 I. 금융사기란?  1. 금융사기의 정의와 특징 ■ 금융사기는 사람을 속이거나 착각하게 하여 금전적 이득을 취하는 범죄 행위 ● 금융사기는 사기 수법과 수단에 따라 양상이 다르게 나타나나, 공통적으로 ‘신뢰  유도-개인정보 수집-금전 편취’ 단계에 걸쳐 금전적 피해를 유발 - 택배회사 발송 배송조회 QR코드(신뢰)로 위장하여 피해자가 코드 스캔 시 악성코드나  앱을 스마트폰에 설치(수집)하고, 이를 통해 탈취한 개인정보로 금전을 유출(탈취) - 유명 핀플루언서(금융+인플루언서)의 영상을 도용(신뢰)하여 가짜 채널로 가

## 9. Retriever 구성

Retriever는 사용자의 질문을 받아 관련 문서 조각을 반환하는 객체이다.

```text
질문 → Retriever → 관련 청크 Top-k 반환
```

In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

retrieved_docs = retriever.invoke("전기통신금융사기와 투자사기의 차이는 무엇인가요?")

for doc in retrieved_docs:
    print(f"page={doc.metadata.get('page_label')}, chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:300].replace("\n", " "))
    print("-" * 80)

page=7, chunk=12
- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹 안심차단(’25.11)을  차례로 시행하여 금융소비자가 자발적 신청을 통해 금융사기를 예방하도록 장려 ● 금융소비자가 직접 신청해야 하는 서비스 외에 자금 이체 시 수취인 계좌에 일정 시간  경과 후 입금되는 지연이체 서비스 등 부정출금 방지를 위한 거래 방식을 도입 표3 | 전기통신금융사기 주요 대응 현황 표4 | 투자사기 주요 대응 현황 시기 주요 내용 시기 주요 내용 ’23.7 통합신고대응센터 개소 ’24.1 자본시장법  개정  Ÿ 불공정거래 
--------------------------------------------------------------------------------
page=5, chunk=7
Hana Bank Hana Institute of Finance 4 2. 금융사기 주요 유형 ■ 금융사기는 금전적 피해를 유발하는 범죄행위 전반을 포괄해 다양한 기준으로  구분되나 전기통신금융사기와 투자사기 두 분류로 구분하는 것이 보편적 ● 전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신을 이용하여 금융기관·    지인·가족 등을 사칭하거나 허위 사실로 피해자의 자금을 탈취하는 수법 - 피싱, 스미싱, 파밍 등이 대표적인 유형이며, 법에서 열거하는 통신수단을 이용하여   개인정보를 노출시켜 자금을 송금·이체하도록 유도 - ’
--------------------------------------------------------------------------------
page=5, chunk=8
사기 행위를 분류하며, 수법에 따라 관련 규정을 지속적으로 확대하는 추세 표2 | 금융사기 분류 그림2 | 금융사기 접근 경로 분류 하위 종류 전기통신 금융사기 보이스피싱 전화로 타인을 사칭하여 송금을 유도하는 수법 메신저피싱 카카오톡이나 SNS로 타인을 사칭하여 송금을  유도하는 수법 파밍 가짜 사이트에 접속하여 개인정보를 입력하도

## 10. RAG 답변 프롬프트 구성

핵심은 LLM이 검색된 문서 밖의 내용을 임의로 생성하지 않도록 제약하는 것이다.

프롬프트 설계 원칙은 다음과 같다.

1. 문서 근거만 사용한다.
2. 문서에서 확인할 수 없으면 모른다고 답한다.
3. 답변 마지막에 근거 페이지를 포함한다.

In [11]:
# 검색 chunk를 context로 변환하는 함수
def format_docs(docs):
    """검색된 문서들을 LLM 입력용 context 문자열로 변환한다."""
    formatted = []
    for doc in docs:
        page = doc.metadata.get("page_label", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        formatted.append(
            f"[출처: page {page}, chunk {chunk_id}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)


# 4. LLM과 프롬프트 초기화
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
당신은 금융사기 보고서 기반 QA 어시스턴트입니다.
아래 [문서]에 근거해서만 답변하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다.'라고 답변하세요.
답변은 핵심 위주로 3~5문장으로 정리하세요.
답변 마지막에 근거 페이지를 '근거: p.숫자' 형식으로 표시하세요.

[문서]
{context}
""".strip()
    ),

    ("human", "{question}"),
])

# MonoRouter는 gpt-4o-mini를 지원하지 않아 gpt-5.4-mini 사용, temperature=0
llm = get_llm("gpt-5.4-mini", temperature=0)

# 5. LCEL Chain 구성(프롬프트, LLM, 문자열 출력 파서)
answer_chain = prompt | llm | StrOutputParser()

In [12]:
question = "전기통신금융사기와 투자사기는 어떻게 다른가요?"

# 질문에 대한 유사도 top 4 검색 하기
docs = vectorstore.similarity_search(question, k=4)

# 검색된 문서를 LLM 입력용 context로 변환
context = format_docs(docs)

# LLM에 질문과 검색된 문서 컨텍스트를 입력하여 답변 생성
answer = answer_chain.invoke({
    "question": question,
    "context": context,
})

In [13]:
# 결과 출력
answer

'전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신 수단을 이용해 금융기관·지인·가족 등을 사칭하거나 허위 사실로 피해자의 자금을 탈취하는 사기입니다. 대표적으로 보이스피싱, 메신저피싱, 파밍, 스미싱, 해킹이 포함됩니다. 반면 투자사기는 수익을 미끼로 투자를 유도하거나 인위적으로 가격을 조작해 금전적 피해를 유발하는 사기이며, 리딩방 사기, 비상장주식 과장 판매, 가짜 HTS/MTS 유도가 예시입니다. 즉, 전기통신금융사기는 사칭·허위정보를 통한 즉각적 자금 탈취가 중심이고, 투자사기는 투자 욕구를 이용한 유인과 수익 조작이 중심입니다. 근거: p.5'

## 11. RAG 함수 만들기

검색과 답변 생성을 하나의 함수로 묶는다.

```text
질문 입력
→ FAISS 검색
→ 검색 문서 format
→ LLM 답변 생성
→ 답변과 출처 반환
```

In [14]:
def answer_question(question: str, k: int = 4):
    """질문에 대해 FAISS 검색 후 문서 기반 답변과 출처 테이블을 반환한다."""
    docs = vectorstore.similarity_search(question, k=k)
    context = format_docs(docs)

    answer = answer_chain.invoke({
        "question": question,
        "context": context,
    })

    sources = pd.DataFrame([
        {
            "rank": i + 1,
            "page": doc.metadata.get("page_label"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "preview": doc.page_content[:120].replace("\n", " ")
        }
        for i, doc in enumerate(docs)
    ])

    return answer, sources

## 12. RAG 실행 예시

다음 질문들은 이 보고서에 포함된 내용을 기반으로 답변 가능한 질문이다.

In [15]:
question = "전기통신금융사기와 투자사기는 어떻게 다른가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)

print("\n[검색된 근거]")
display(sources)  # Jupyter 환경에서는 display로 테이블이 예쁘게 나옵니다.


[질문]
전기통신금융사기와 투자사기는 어떻게 다른가요?

[답변]
전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신 수단을 이용해 금융기관·지인·가족 등을 사칭하거나 허위 사실로 피해자의 자금을 탈취하는 사기입니다. 반면 투자사기는 투자자의 수익 기대를 이용해 투자를 유도하거나 인위적으로 가격을 조작해 금전적 피해를 유발하는 범죄입니다. 즉, 전기통신금융사기는 ‘사칭·기망을 통한 직접 탈취’에, 투자사기는 ‘투자 유인과 수익 기대 악용’에 초점이 있습니다. 또한 전기통신금융사기는 보이스피싱·스미싱·파밍 등이 대표적이고, 투자사기는 리딩방 사기·비상장주식 과장 판매·가짜 HTS/MTS 등이 대표적입니다. 근거: p.5

[검색된 근거]


,rank,page,chunk_id,preview
0,1,5,7,Hana Bank Hana Institute of Finance 4 2. 금융사기 ...
1,2,5,8,"사기 행위를 분류하며, 수법에 따라 관련 규정을 지속적으로 확대하는 추세 표2 | ..."
2,3,7,12,"- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹..."
3,4,3,4,측면에서 금융회사와 정부 기관의 예방 노력이 함께 이행되어야 할 필요 l 개별법 제...


In [16]:
question = "금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?

[답변]
금융회사들은 전자금융거래 패턴을 실시간으로 분석해 부정 거래를 탐지·차단하는 이상금융거래탐지시스템(FDS)을 도입하고 있습니다. 또한 거래 내역, 고객 정보, 평소 거래 패턴 등을 분석해 이상 거래를 막고, 사기 방지를 위한 소프트웨어도 자체 개발하고 있습니다. 최근에는 AI와 생성형 AI를 접목해 신종 사기 수법을 학습하고, 고객 특성별 거래 유형이나 이상 행동 패턴을 탐지해 경고 문구를 표시하는 방식도 추진하고 있습니다. 통신사 등과의 협업을 통해 통화 중 이체 시 사기 가능성을 알리는 실시간 알림 체계도 마련하고 있습니다. 근거: p.9

[검색된 근거]


,rank,page,chunk_id,preview
0,1,13,24,"● 금융거래 데이터뿐만 아니라 앱 이용 패턴 데이터, 통신 데이터 등 사기 거래 전..."
1,2,3,4,측면에서 금융회사와 정부 기관의 예방 노력이 함께 이행되어야 할 필요 l 개별법 제...
2,3,9,15,Hana Bank Hana Institute of Finance 8 2. 금융회사 ...
3,4,12,22,"- 거래 이력, 계좌 이름, 결제 금액 등 사기 계좌를 탐지할 수 있는 정보를 종합..."


## 13. 문서에 없는 질문 테스트

RAG 실습에서는 답변 가능한 질문만 넣으면 안 된다.  
문서에 없는 질문을 넣어 **모른다고 답하는지** 확인해야 한다.

In [17]:
question = "2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?

[답변]
문서에서 확인할 수 있는 최신 수치는 **’24년 기준 보이스피싱·유사수신 등 불특정 다수 대상 금융사기 피해금액 약 1.7조 원**입니다. 다만 사용자가 पूछ은 **2026년 6월 기준** 피해 금액은 문서에 제시되어 있지 않아 확인할 수 없습니다. 또한 문서에는 ’25년 10월 기준 자율배상 신청 현황만 있으며, 보이스피싱 전체 피해금액의 2026년 6월 최신치는 없습니다.  
근거: p.6, p.10

[검색된 근거]


,rank,page,chunk_id,preview
0,1,6,10,● 연애 관계를 빙자하여 자금을 탈취하는 로맨스스캠의 경우 ’24년 2~12월 기준...
1,2,10,18,- 금융사기의 수법이 진화하여 소비자 개인이 사기를 예방하는 데 실질적 한계가 있다...
2,3,6,9,하나은행 ...
3,4,10,17,하나은행 ...


## 14. 정리

이 실습에서 구현한 Naive RAG 흐름은 다음과 같다.

```text
PDF 로딩
→ 페이지 단위 Document 생성
→ RecursiveCharacterTextSplitter로 청킹
→ OpenAIEmbeddings로 임베딩
→ FAISS 벡터스토어 생성
→ 질문 기반 Top-k 검색
→ 검색 문서 기반 LLM 답변 생성
```

## 핵심 포인

| 관찰 포인트 | 핵심 내용 |
|---|---|
| 검색 결과가 부정확함 | 청크 크기, 질문 표현, Top-k 조정 필요 |
| 답변이 문서 밖 내용을 포함함 | 프롬프트 제약 및 문서 없음 질문 테스트 필요 |
| 표·그림 내용이 누락됨 | PDF 파싱 방식의 한계, Markdown 변환 비교 필요 |
| 같은 질문인데 답변이 달라짐 | 검색 결과와 LLM 생성 조건을 분리해서 점검해야 함 |
| 인덱스 재사용 필요 | FAISS 저장·로드 구조로 인덱싱 비용 절감 가능 |

## 15. 개선 실험

위 결과에서 관찰한 문제 3가지를 하나씩 개선해 본다.

| 문제 | 관찰 위치 | 개선 방법 |
|---|---|---|
| 청크에 페이지 머리글(`하나은행 ... 하나금융연구소`)과 긴 공백이 섞임 | 6. 청크 분포 확인 | 청킹 전 텍스트 전처리 |
| 정답 청크가 1등으로 검색되지 않음 | 8. 기본 유사도 검색 | 청크 크기 비교 실험 |
| 문서에 없는 질문에 부가 정보를 덧붙이고, 외국어가 섞임 | 13. 문서에 없는 질문 | 프롬프트 규칙 강화 |

### 15-1. 텍스트 전처리

PDF 페이지마다 머리글이 반복된다.

- 홀수 페이지: `하나은행` + 긴 공백 + `하나금융연구소 33`
- 짝수 페이지: `Hana Bank` / `Hana Institute of Finance` / `4`

머리글은 질문과 무관한데 모든 청크에 들어가 임베딩을 흐리게 만들므로 청킹 전에 제거한다.

In [18]:
import re
import copy

def clean_text(text):
    """페이지 머리글과 연속 공백을 제거한다."""
    text = re.sub(r"^\s*하나은행\s+하나금융연구소\s*\d+\s*\n", "", text)
    text = re.sub(r"^\s*Hana Bank\s*\nHana Institute of Finance\s*\n\d+\s*\n", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

# 원본 pages는 그대로 두고 복사본에 전처리 적용
clean_pages = copy.deepcopy(pages)
for doc in clean_pages:
    doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1
    doc.page_content = clean_text(doc.page_content)

print("[전처리 전]")
print(repr(pages[3].page_content[:150]))
print("\n[전처리 후]")
print(repr(clean_pages[3].page_content[:150]))

[전처리 전]
'하나은행\n                                                                                                                   하나금융연구소 33\nI. 금융사기란? \n1. 금융사기의'

[전처리 후]
'I. 금융사기란? \n1. 금융사기의 정의와 특징\n■ 금융사기는 사람을 속이거나 착각하게 하여 금전적 이득을 취하는 범죄 행위\n● 금융사기는 사기 수법과 수단에 따라 양상이 다르게 나타나나, 공통적으로 ‘신뢰 \n유도-개인정보 수집-금전 편취’ 단계에 걸쳐 금전적 피해를 '


### 15-2. 청크 크기 비교 실험

정답 페이지를 알고 있는 질문 8개로 평가셋을 만들고, **전처리 여부 × 청크 크기** 조합별로 검색 성능을 비교한다.

- `hit@1`: 정답 페이지가 1등으로 검색된 질문 수
- `hit@4`: 정답 페이지가 상위 4개 안에 들어간 질문 수
- `MRR`: 정답 순위의 역수 평균 (1등이면 1, 2등이면 0.5 …, 1에 가까울수록 좋음)

In [19]:
# (질문, 정답 페이지 집합)
EVAL_SET = [
    ("금융사기는 어떤 단계로 발생하나요?", {4}),
    ("전기통신금융사기와 투자사기는 어떻게 다른가요?", {5}),
    ("최근 2년 내 금융사기에 노출된 금융소비자 비율은?", {6}),
    ("1인당 평균 피해액은 얼마인가요?", {6}),
    ("지연이체 서비스란 무엇인가요?", {7}),
    ("이상금융거래탐지시스템(FDS)은 무엇인가요?", {9}),
    ("보이스피싱 피해를 금융회사가 배상하면 배상액 규모는 얼마인가요?", {10}),
    ("해외에서는 금융사기에 어떻게 대응하고 있나요?", {11, 12}),
]

def build_store(docs, chunk_size, chunk_overlap):
    """문서를 청킹해서 메모리 FAISS 벡터스토어를 만든다."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    split_docs = splitter.split_documents(docs)
    for i, doc in enumerate(split_docs):
        doc.metadata["chunk_id"] = i
        doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1
    return FAISS.from_documents(split_docs, embeddings), len(split_docs)

def evaluate(store, k=4):
    """평가셋 질문마다 정답 페이지의 검색 순위를 구한다."""
    ranks = []
    for q, gold_pages in EVAL_SET:
        found = [d.metadata["page_label"] for d in store.similarity_search(q, k=k)]
        rank = next((i + 1 for i, page in enumerate(found) if page in gold_pages), None)
        ranks.append(rank)
    return ranks

rows = []
for use_clean, docs in [(False, pages), (True, clean_pages)]:
    for size, overlap in [(800, 120), (500, 80), (300, 50)]:
        store, n_chunks = build_store(docs, size, overlap)
        ranks = evaluate(store)
        rows.append({
            "전처리": use_clean,
            "chunk_size": size,
            "chunk_overlap": overlap,
            "청크 수": n_chunks,
            "hit@1": sum(r == 1 for r in ranks),
            "hit@4": sum(r is not None for r in ranks),
            "MRR": round(sum(1 / r for r in ranks if r) / len(ranks), 3),
            "질문별 순위": [r or "-" for r in ranks],
        })

display(pd.DataFrame(rows))

,전처리,chunk_size,chunk_overlap,청크 수,hit@1,hit@4,MRR,질문별 순위
0,False,800,120,27,6,8,0.875,"[2, 1, 1, 1, 1, 1, 1, 2]"
1,False,500,80,40,7,8,0.938,"[1, 1, 1, 1, 2, 1, 1, 1]"
2,False,300,50,64,7,8,0.938,"[1, 2, 1, 1, 1, 1, 1, 1]"
3,True,800,120,27,7,8,0.938,"[1, 2, 1, 1, 1, 1, 1, 1]"
4,True,500,80,35,6,8,0.844,"[1, 1, 1, 1, 2, 1, 1, 4]"
5,True,300,50,60,8,8,1.000,"[1, 1, 1, 1, 1, 1, 1, 1]"


> [해석] 직접 실행했을 때는 **전처리 + chunk_size=300**이 8문제 모두 정답 페이지를 1등으로 찾았다 (원래 설정 800/120은 hit@1 6/8).
> 다만 평가 질문이 8개뿐이라 조합 간 차이가 1~2문제 수준이므로, 실제 서비스라면 평가셋을 늘려서 다시 확인해야 한다.

### 15-3. 프롬프트 규칙 강화

기존 프롬프트는 "문서에서 확인할 수 없는 내용은 … 라고 답변하세요"라고만 했기 때문에,
LLM이 **비슷한 다른 시점의 수치**('24년 1.7조 원 등)를 덧붙이거나 근거 페이지를 붙이는 문제가 있었다.

강화한 규칙:
1. 사전 지식으로 보충 금지
2. 답이 없으면 **그 한 문장만** 출력 (근거 표시도 하지 않음)
3. 다른 시점·대상의 정보로 대신 답하지 않기
4. 한국어로만 답변

In [20]:
improved_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
당신은 금융사기 보고서 기반 QA 어시스턴트입니다.

[규칙]
1. 아래 [문서]에 적힌 내용만 근거로 답변하세요. 사전 지식으로 내용을 보충하지 마세요.
2. 질문이 묻는 답(시점·대상·수치 포함)이 [문서]에 없으면, 다른 설명이나 근거 표시 없이 정확히 '문서에서 확인할 수 없습니다.' 한 문장만 출력하세요.
3. 비슷한 다른 시점이나 다른 대상의 정보로 대신 답하지 마세요.
4. 답변은 반드시 한국어로만 작성하세요.
5. 답변은 핵심 위주로 3~5문장으로 정리하세요.
6. [문서]로 답변한 경우에만 마지막 줄에 근거 페이지를 '근거: p.숫자' 형식으로 표시하세요.

[문서]
{context}
""".strip()
    ),
    ("human", "{question}"),
])

improved_chain = improved_prompt | llm | StrOutputParser()

# 실험 결과가 가장 좋았던 설정(전처리 + 300/50)으로 벡터스토어 구성
improved_store, _ = build_store(clean_pages, chunk_size=300, chunk_overlap=50)

In [21]:
NO_ANSWER = "문서에서 확인할 수 없습니다."

# (질문, 문서로 답변 가능한지)
test_questions = [
    ("2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?", False),
    ("하나은행의 2025년 당기순이익은 얼마인가요?", False),
    ("금융사기 피해를 입으면 경찰 신고는 몇 번으로 하나요?", False),
    ("전기통신금융사기와 투자사기는 어떻게 다른가요?", True),
    ("보이스피싱 피해를 금융회사가 배상하면 배상액 규모는 얼마인가요?", True),
]

def compare_prompts(question):
    """같은 검색 결과에 기존/개선 프롬프트를 적용해 답변을 비교한다."""
    context = format_docs(improved_store.similarity_search(question, k=4))
    old = answer_chain.invoke({"question": question, "context": context})
    new = improved_chain.invoke({"question": question, "context": context})
    return old, new

results = []
for q, answerable in test_questions:
    old, new = compare_prompts(q)
    check = (lambda a: NO_ANSWER not in a) if answerable else (lambda a: a.strip() == NO_ANSWER)
    results.append({
        "질문": q,
        "답변 가능": answerable,
        "기존 프롬프트": "OK" if check(old) else "BAD",
        "개선 프롬프트": "OK" if check(new) else "BAD",
    })
    print(f"[질문] {q}")
    print(f"  [기존] {old}")
    print(f"  [개선] {new}")
    print("-" * 80)

display(pd.DataFrame(results))

[질문] 2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?
  [기존] 문서에는 2026년 6월 기준 최신 보이스피싱 피해 금액이 직접 제시되어 있지 않습니다. 다만 문서에는 보이스피싱을 포함한 불특정 다수 대상 금융사기 피해 금액이 ’24년 약 1.7조 원으로 증가했다고만 나와 있습니다. 따라서 질문하신 “2026년 6월 기준 최신” 수치는 문서에서 확인할 수 없습니다. 근거: p.6
  [개선] 문서에서 확인할 수 없습니다.
--------------------------------------------------------------------------------
[질문] 하나은행의 2025년 당기순이익은 얼마인가요?
  [기존] 문서에서 하나은행의 2025년 당기순이익은 확인할 수 없습니다. 제공된 문서는 보이스피싱 피해 배상, 금융사기 예방 대책, 관련 제도 개정 내용 위주로 되어 있어 하나은행의 당기순이익 수치를 포함하고 있지 않습니다. 따라서 해당 금액은 문서에서 확인할 수 없습니다. 근거: p.7, p.10
  [개선] 문서에서 확인할 수 없습니다.
--------------------------------------------------------------------------------
[질문] 금융사기 피해를 입으면 경찰 신고는 몇 번으로 하나요?
  [기존] 문서에서는 금융사기 피해 시 경찰 신고 번호가 무엇인지 확인할 수 없습니다.  
제시된 문서는 피해 현황과 사기 유형, 피해 규모에 대한 내용만 포함하고 있습니다.  
따라서 문서에서 확인할 수 없는 내용은 답변드릴 수 없습니다.  
근거: p.2, p.6
  [개선] 문서에서 확인할 수 없습니다.
--------------------------------------------------------------------------------
[질문] 전기통신금융사기와 투자사기는 어떻게 다른가요?
  [기존] 전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신 수단을 

,질문,답변 가능,기존 프롬프트,개선 프롬프트
0,2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?,False,BAD,OK
1,하나은행의 2025년 당기순이익은 얼마인가요?,False,BAD,OK
2,금융사기 피해를 입으면 경찰 신고는 몇 번으로 하나요?,False,BAD,OK
3,전기통신금융사기와 투자사기는 어떻게 다른가요?,True,OK,OK
4,보이스피싱 피해를 금융회사가 배상하면 배상액 규모는 얼마인가요?,True,OK,OK


> [해석] 기존 프롬프트는 문서에 없는 질문 3개 모두에서 "확인할 수 없다"고 하면서도 다른 수치를 덧붙이거나 근거 페이지를 붙였다.
> 개선 프롬프트는 3개 모두 정확히 한 문장으로 거절했고, 답변 가능한 질문 2개는 그대로 잘 답했다.

## 16. 개선 결과 정리

| 개선 항목 | 변경 내용 | 효과 |
|---|---|---|
| 전처리 | 페이지 머리글, 연속 공백 제거 | 청크에서 검색과 무관한 노이즈 제거 |
| 청킹 | 800/120 → 300/50 | 평가셋 hit@1 6/8 → 8/8 |
| 프롬프트 | 거절 형식·언어·대체 정보 금지 규칙 추가 | 문서 밖 질문 거절 0/3 → 3/3 |